# Genre Classification Visualization

Comprehensive visualizations of genre classification results from both Antares and NewMag pipelines.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

## Load Data from Pipeline Outputs

Load genre classification results from both Antares and NewMag pipelines.

In [ ]:
# Load data from both pipelines
antares_df = pd.read_csv('antares/antares_genres_output.csv')
newmag_df = pd.read_csv('newmag/newmag_genres_output.csv')

# Display first few rows and info
print("=== ANTARES DATA ===")
print(f"Shape: {antares_df.shape}")
print(antares_df.head(10))
print("\n=== NEWMAG DATA ===")
print(f"Shape: {newmag_df.shape}")
print(newmag_df.head(10))

## Parse and Aggregate Genre Data

Extract genre counts from the last rows (Summary) of each dataset.

In [ ]:
# Function to parse top genres from summary text
import re

def parse_summary_genres(summary_text):
    """Extract top genres from the summary row text"""
    try:
        # Extract from format: "Top genres by title count: [('Genre', count), ...]"
        match = re.search(r"Top genres by title count: \[(.*?)\]", str(summary_text))
        if match:
            genres_str = match.group(1)
            # Parse tuples
            pattern = r"\('([^']+)',\s*(\d+)\)"
            matches = re.findall(pattern, genres_str)
            return {genre: int(count) for genre, count in matches}
    except Exception as e:
        print(f"Error parsing: {e}")
    return {}

# Extract genre data
antares_summary = antares_df[antares_df['Title'] == 'SUMMARY'].iloc[0] if any(antares_df['Title'] == 'SUMMARY') else None
newmag_summary = newmag_df[newmag_df['Title'] == 'SUMMARY'].iloc[0] if any(newmag_df['Title'] == 'SUMMARY') else None

print("Antares Summary:", antares_summary['Genres'] if antares_summary is not None else "Not found")
print("\nNewMag Summary:", newmag_summary['Genres'] if newmag_summary is not None else "Not found")

# Get non-summary data for analysis
antares_data = antares_df[antares_df['Title'] != 'SUMMARY'].dropna()
newmag_data = newmag_df[newmag_df['Title'] != 'SUMMARY'].dropna()

print(f"\nAntares valid records: {len(antares_data)}")
print(f"NewMag valid records: {len(newmag_data)}")

## Genre Distribution: Top Genres by Title Count

Bar charts comparing the most common genres across both datasets.

In [ ]:
# Function to extract genre counts from comma-separated genre strings
def count_genres(df, dataset_name):
    """Count occurrences of each genre in a dataset"""
    genre_counts = {}
    for genres_str in df['Genres'].dropna():
        if isinstance(genres_str, str) and genres_str.strip():
            genres = [g.strip() for g in str(genres_str).split(',')]
            for genre in genres:
                if genre:
                    genre_counts[genre] = genre_counts.get(genre, 0) + 1
    return genre_counts

# Count genres
antares_genres = count_genres(antares_data, "Antares")
newmag_genres = count_genres(newmag_data, "NewMag")

# Sort by count
antares_sorted = dict(sorted(antares_genres.items(), key=lambda x: x[1], reverse=True))
newmag_sorted = dict(sorted(newmag_genres.items(), key=lambda x: x[1], reverse=True))

print("TOP 5 ANTARES GENRES:")
for genre, count in list(antares_sorted.items())[:5]:
    print(f"  {genre}: {count}")

print("\nTOP 5 NEWMAG GENRES:")
for genre, count in list(newmag_sorted.items())[:5]:
    print(f"  {genre}: {count}")

In [ ]:
# Create side-by-side bar charts
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Antares top 8 genres
antares_top = dict(list(antares_sorted.items())[:8])
axes[0].barh(list(antares_top.keys()), list(antares_top.values()), color='steelblue')
axes[0].set_xlabel('Count', fontsize=12, fontweight='bold')
axes[0].set_title('Top Genres in Antares Dataset', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()
for i, v in enumerate(antares_top.values()):
    axes[0].text(v + 0.5, i, str(v), va='center', fontweight='bold')

# NewMag top 8 genres
newmag_top = dict(list(newmag_sorted.items())[:8])
axes[1].barh(list(newmag_top.keys()), list(newmag_top.values()), color='coral')
axes[1].set_xlabel('Count', fontsize=12, fontweight='bold')
axes[1].set_title('Top Genres in NewMag Dataset', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()
for i, v in enumerate(newmag_top.values()):
    axes[1].text(v + 0.5, i, str(v), va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('genre_distribution_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: genre_distribution_comparison.png")

## Language Distribution Analysis

Visualize the distribution of detected languages in both datasets.

In [ ]:
# Language distribution
antares_langs = antares_data['Language'].value_counts()
newmag_langs = newmag_data['Language'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Antares
antares_langs.head(10).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Count', fontsize=12, fontweight='bold')
axes[0].set_title(f'Language Distribution (Antares) - {len(antares_data)} titles', 
                   fontsize=14, fontweight='bold')
axes[0].invert_yaxis()

# NewMag
newmag_langs.head(10).plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_xlabel('Count', fontsize=12, fontweight='bold')
axes[1].set_title(f'Language Distribution (NewMag) - {len(newmag_data)} titles',
                  fontsize=14, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('language_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: language_distribution.png")
print(f"\nAntares languages: {len(antares_langs)} unique")
print(f"NewMag languages: {len(newmag_langs)} unique")

## Genre Volume Distribution

Analyze total units/sales/count by genre (weighted by the Number column).

In [ ]:
# Calculate volume (Number) distribution by genre
def get_genre_volumes(df, dataset_name):
    """Sum the Number column for each genre"""
    genre_volumes = {}
    for idx, row in df.iterrows():
        genres_str = row['Genres']
        number = float(row['Number']) if pd.notna(row['Number']) else 0
        if isinstance(genres_str, str) and genres_str.strip():
            genres = [g.strip() for g in str(genres_str).split(',')]
            for genre in genres:
                if genre:
                    # Distribute volume equally among genres
                    genre_volumes[genre] = genre_volumes.get(genre, 0) + (number / len(genres))
    return genre_volumes

antares_volumes = get_genre_volumes(antares_data, "Antares")
newmag_volumes = get_genre_volumes(newmag_data, "NewMag")

# Sort by volume
antares_vol_sorted = dict(sorted(antares_volumes.items(), key=lambda x: x[1], reverse=True))
newmag_vol_sorted = dict(sorted(newmag_volumes.items(), key=lambda x: x[1], reverse=True))

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Antares volume
antares_vol_top = dict(list(antares_vol_sorted.items())[:8])
axes[0].barh(list(antares_vol_top.keys()), list(antares_vol_top.values()), color='darkblue')
axes[0].set_xlabel('Total Volume', fontsize=12, fontweight='bold')
axes[0].set_title('Top Genres by Volume (Antares)', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()
for i, v in enumerate(antares_vol_top.values()):
    axes[0].text(v + 100, i, f'{int(v)}', va='center', fontweight='bold')

# NewMag volume
newmag_vol_top = dict(list(newmag_vol_sorted.items())[:8])
axes[1].barh(list(newmag_vol_top.keys()), list(newmag_vol_top.values()), color='darkred')
axes[1].set_xlabel('Total Volume', fontsize=12, fontweight='bold')
axes[1].set_title('Top Genres by Volume (NewMag)', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()
for i, v in enumerate(newmag_vol_top.values()):
    axes[1].text(v + 100, i, f'{int(v)}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('genre_volume_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: genre_volume_distribution.png")

## Pie Charts: Genre Composition

Visualize the proportional contribution of each genre to the total dataset.

In [ ]:
# Pie charts for genre composition
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Antares pie
antares_top_pie = dict(list(antares_sorted.items())[:6])
antares_other = len(antares_genres) - sum(antares_top_pie.values())
if antares_other > 0:
    antares_top_pie['Other'] = antares_other
colors_antares = sns.color_palette('Blues', len(antares_top_pie))
axes[0].pie(antares_top_pie.values(), labels=antares_top_pie.keys(), 
            autopct='%1.1f%%', colors=colors_antares, startangle=90)
axes[0].set_title(f'Genre Composition (Antares) - {len(antares_data)} titles',
                  fontsize=14, fontweight='bold')

# NewMag pie
newmag_top_pie = dict(list(newmag_sorted.items())[:6])
newmag_other = len(newmag_genres) - sum(newmag_top_pie.values())
if newmag_other > 0:
    newmag_top_pie['Other'] = newmag_other
colors_newmag = sns.color_palette('Reds', len(newmag_top_pie))
axes[1].pie(newmag_top_pie.values(), labels=newmag_top_pie.keys(),
            autopct='%1.1f%%', colors=colors_newmag, startangle=90)
axes[1].set_title(f'Genre Composition (NewMag) - {len(newmag_data)} titles',
                  fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('genre_composition_pie.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: genre_composition_pie.png")

## Summary Statistics

Key metrics and insights from both datasets.

In [ ]:
# Create summary statistics
summary_stats = {
    'Metric': [
        'Total Titles',
        'Unique Genres',
        'Unique Languages',
        'Total Volume',
        'Avg Volume/Title',
        'Top Genre',
        'Top Genre Count',
    ],
    'Antares': [
        len(antares_data),
        len(antares_genres),
        len(antares_langs),
        int(antares_data['Number'].sum()),
        int(antares_data['Number'].mean()),
        list(antares_sorted.keys())[0] if antares_sorted else 'N/A',
        antares_sorted[list(antares_sorted.keys())[0]] if antares_sorted else 0,
    ],
    'NewMag': [
        len(newmag_data),
        len(newmag_genres),
        len(newmag_langs),
        int(newmag_data['Number'].sum()),
        int(newmag_data['Number'].mean()),
        list(newmag_sorted.keys())[0] if newmag_sorted else 'N/A',
        newmag_sorted[list(newmag_sorted.keys())[0]] if newmag_sorted else 0,
    ]
}

summary_df = pd.DataFrame(summary_stats)

# Display table
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=summary_df.values, colLabels=summary_df.columns,
                cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2)

# Style header
for i in range(len(summary_df.columns)):
    table[(0, i)].set_facecolor('#4472C4')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Alternate row colors
for i in range(1, len(summary_df) + 1):
    for j in range(len(summary_df.columns)):
        if i % 2 == 0:
            table[(i, j)].set_facecolor('#E7E6E6')
        else:
            table[(i, j)].set_facecolor('#F2F2F2')

plt.title('Summary Statistics: Antares vs NewMag', fontsize=14, fontweight='bold', pad=20)
plt.savefig('summary_statistics.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: summary_statistics.png")
print("\n" + "="*60)
print(summary_df.to_string(index=False))
print("="*60)

## Top 8 Genres Comparison

Direct comparison of the top 8 genres between Antares and NewMag using a grouped bar chart.

In [ ]:
# Get all unique genres from top 8 of each
all_genres = set(list(antares_sorted.keys())[:8] + list(newmag_sorted.keys())[:8])

# Create comparison data
comparison_data = []
for genre in sorted(all_genres):
    comparison_data.append({
        'Genre': genre,
        'Antares': antares_sorted.get(genre, 0),
        'NewMag': newmag_sorted.get(genre, 0)
    })

comparison_df = pd.DataFrame(comparison_data).sort_values('Antares', ascending=True)

# Create grouped bar chart
fig, ax = plt.subplots(figsize=(14, 8))
x = np.arange(len(comparison_df))
width = 0.35

bars1 = ax.barh(x - width/2, comparison_df['Antares'], width, label='Antares', color='steelblue')
bars2 = ax.barh(x + width/2, comparison_df['NewMag'], width, label='NewMag', color='coral')

ax.set_ylabel('Genre', fontsize=12, fontweight='bold')
ax.set_xlabel('Title Count', fontsize=12, fontweight='bold')
ax.set_title('Genre Comparison: Antares vs NewMag', fontsize=14, fontweight='bold')
ax.set_yticks(x)
ax.set_yticklabels(comparison_df['Genre'])
ax.legend(fontsize=11)
ax.grid(axis='x', alpha=0.3)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        width_val = bar.get_width()
        if width_val > 0:
            ax.text(width_val, bar.get_y() + bar.get_height()/2, 
                   f'{int(width_val)}', va='center', ha='left', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('genre_comparison_grouped.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: genre_comparison_grouped.png")

## Key Insights & Summary

**Generated Files:**
- `genre_distribution_comparison.png` - Top genres by title count
- `language_distribution.png` - Language distribution across both datasets
- `genre_volume_distribution.png` - Total volume/units by genre
- `genre_composition_pie.png` - Proportional genre composition
- `summary_statistics.png` - Key metrics table
- `genre_comparison_grouped.png` - Direct genre comparison

**Next Steps:**
All charts have been saved in the `book-genre-ai/` directory and can be used in reports, presentations, or further analysis.